# Index · Unit 9 — Learning & adaptation

The final unit. Same shape as Units 1–8. For every component in **Section 10 of
`index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python
  (no API key needed) — here "from scratch" means a real (tiny) learning loop.
- **SOTA** — how it's actually done today (fine-tuning API, DPO/TRL, self-refine,
  skill libraries). Some need `OPENAI_API_KEY`.

Each component lists **📚 Resources** (also consolidated in `RESOURCES.md`).

Components: **1** fine-tuning / instruction tuning · **2** RLHF / preference
optimization · **3** self-correction / critique loops · **4** skill acquisition /
memory-based learning.

**Intuition for the whole unit:** every unit so far kept the model *fixed* and
built scaffolding around it. Learning is about **getting better over time** —
either by changing the **weights** (fine-tuning, RLHF: slow, permanent, offline)
or by changing the **context** (self-correction, stored skills: fast, per-run, no
training). Know which axis you're on: weights vs. context.

## Setup

In [ ]:
import os, re, json, math

MODEL = "gpt-4o-mini"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

def sigmoid(z): return 1 / (1 + math.exp(-z))

## 1. Fine-tuning / instruction tuning

Fine-tuning **updates the model's weights** on your examples so it internalizes a
behavior or format — no need to re-explain it in every prompt. The core mechanism
is **gradient descent**: show (input → desired output) pairs, measure error, nudge
the weights to reduce it, repeat. Instruction tuning is the same, on
(instruction → response) data.

**From scratch:** a real gradient-descent loop training a tiny classifier — watch
the weights adapt to the data.

> 📚 **Resources:** [OpenAI — Fine-tuning guide](https://platform.openai.com/docs/guides/fine-tuning) ·
> [InstructGPT — Ouyang et al., 2022](https://arxiv.org/abs/2203.02155)

In [ ]:
# From scratch — train a tiny sentiment classifier by gradient descent.
POS, NEG = {"great", "good", "love", "excellent"}, {"awful", "bad", "terrible", "hate"}
def features(text):
    toks = text.lower().split()
    return [sum(w in POS for w in toks), sum(w in NEG for w in toks)]

DATA = [("great movie", 1), ("awful film", 0), ("i love this", 1),
        ("terrible and bad", 0), ("good good", 1), ("hate it", 0)]

w, b, lr = [0.0, 0.0], 0.0, 0.3
for epoch in range(300):                                   # the learning loop
    for text, y in DATA:
        f = features(text)
        p = sigmoid(w[0]*f[0] + w[1]*f[1] + b)            # prediction
        g = p - y                                          # error gradient
        w[0] -= lr*g*f[0]; w[1] -= lr*g*f[1]; b -= lr*g   # nudge weights

def predict(text):
    f = features(text); return round(sigmoid(w[0]*f[0] + w[1]*f[1] + b), 2)

print(f"learned weights: pos={w[0]:.2f}, neg={w[1]:.2f} (pos↑ good, neg↓ good)")
print("new 'excellent work' ->", predict("excellent work"))   # generalizes to unseen words
print("new 'hate hate'      ->", predict("hate hate"))

In [ ]:
# SOTA — fine-tune a hosted model on a chat-format JSONL dataset.
# Pattern (pseudo — needs a real uploaded file + minutes of training):
#
#   # training.jsonl — one example per line:
#   # {"messages": [{"role":"system","content":"You are terse."},
#   #               {"role":"user","content":"Summarize: ..."},
#   #               {"role":"assistant","content":"<ideal terse summary>"}]}
#   f = OpenAI().files.create(file=open("training.jsonl","rb"), purpose="fine-tune")
#   job = OpenAI().fine_tuning.jobs.create(training_file=f.id, model="gpt-4o-mini-2024-07-18")
#   # ...poll job until done, then call the resulting model id like any other model.
example = {"messages": [{"role": "user", "content": "ping"},
                        {"role": "assistant", "content": "pong"}]}
print("one JSONL training row:", json.dumps(example))
print("same gradient-descent idea as above — just billions of weights, managed for you.")

## 2. RLHF / preference optimization

Often you can't write the "correct" answer, but you *can* say **A is better than
B**. RLHF learns from such **preferences**: train a **reward model** to score
outputs so preferred > rejected, then optimize the policy toward high reward.
**DPO** skips the separate RL step and optimizes the policy directly on preference
pairs.

**From scratch:** a reward model fit from pairwise preferences (Bradley–Terry
update), then rank candidates by learned reward.

> 📚 **Resources:** [DPO — Rafailov et al., 2023](https://arxiv.org/abs/2305.18290) ·
> [HuggingFace TRL](https://github.com/huggingface/trl) ·
> [InstructGPT (RLHF) — Ouyang et al., 2022](https://arxiv.org/abs/2203.02155)

In [ ]:
# From scratch — learn a reward from "winner beats loser" pairs (Bradley-Terry).
CANDIDATES = ["detailed and correct", "short but correct", "confidently wrong", "off topic"]
PREFS = [("detailed and correct", "short but correct"),
         ("short but correct", "confidently wrong"),
         ("confidently wrong", "off topic"),
         ("detailed and correct", "off topic")]

reward = {c: 0.0 for c in CANDIDATES}
lr = 0.3
for _ in range(500):                                       # fit rewards to preferences
    for win, lose in PREFS:
        p = sigmoid(reward[win] - reward[lose])            # P(win preferred)
        reward[win]  += lr * (1 - p)                       # push winner up
        reward[lose] -= lr * (1 - p)                       # push loser down

for c, r in sorted(reward.items(), key=lambda kv: -kv[1]):
    print(f"  reward {r:+.2f}  {c}")
best = max(reward, key=reward.get)
print("policy would pick ->", best)

In [ ]:
# SOTA — preference fine-tuning (DPO). Dataset = prompt + chosen + rejected.
# Pattern (pseudo — TRL's DPOTrainer, or OpenAI preference fine-tuning):
#
#   # prefs.jsonl: {"input":{"messages":[...]},
#   #               "preferred_output":[{"role":"assistant","content":"<better>"}],
#   #               "non_preferred_output":[{"role":"assistant","content":"<worse>"}]}
#   # OpenAI: fine_tuning.jobs.create(..., method={"type":"dpo"})
#   # TRL:    DPOTrainer(model, train_dataset=prefs, ...).train()
pref_row = {"prompt": "Explain RAG.",
            "chosen": "Retrieval-augmented generation grounds answers in fetched docs.",
            "rejected": "RAG is a kind of fabric."}
print("one preference row:", json.dumps(pref_row))
print("same winner>loser signal as above — optimized directly into the weights.")

## 3. Self-correction / critique loops

Learning without any training: have the model **critique and revise its own
output** until it passes a check. (Reflexion in Unit 1 §4 used this for reasoning;
here it's the general *improvement* loop.) It adapts **per run** via the context,
not the weights — cheap and immediate, but gone when the session ends.

**From scratch:** draft → check → revise, looping until a validator passes.

> 📚 **Resources:** [Self-Refine — Madaan et al., 2023](https://arxiv.org/abs/2303.17651) ·
> [Reflexion — Shinn et al., 2023](https://arxiv.org/abs/2303.11366)

In [ ]:
# From scratch — iterative refine until a checker passes.
REQUIRED = {"cost", "latency"}                 # the answer must mention both

def checker(answer):
    missing = REQUIRED - set(answer.lower().split())
    return (not missing), missing

def fake_reviser(answer, missing):
    return (answer + " " + " ".join(missing)).strip()     # add what's missing

def self_correct(draft, max_iters=3):
    answer = draft
    for i in range(max_iters):
        ok, missing = checker(answer)
        print(f"  iter {i}: {'PASS' if ok else 'missing ' + str(missing)} -> '{answer}'")
        if ok: return answer
        answer = fake_reviser(answer, missing)             # critique -> revise
    return answer

self_correct("optimize for cost")

In [ ]:
# SOTA — model critiques its own draft, then rewrites; loop until a judge approves.
def sota_self_refine(task, max_iters=3):
    draft = chat([{"role": "user", "content": task}]).content
    for i in range(max_iters):
        critique = chat([
            {"role": "system", "content": "Critique this answer. If it is excellent reply exactly "
             "'APPROVED'. Otherwise give one concrete improvement."},
            {"role": "user", "content": f"Task: {task}\nAnswer: {draft}"}]).content
        print(f"  iter {i}: {critique[:60]}")
        if "APPROVED" in critique: break
        draft = chat([{"role": "user", "content":
                       f"Task: {task}\nDraft: {draft}\nImprove it using: {critique}"}]).content
    return draft

print(sota_self_refine("Give one tip for reliable agents.", max_iters=2))

## 4. Skill acquisition / memory-based learning

The agent **gets better across runs** by saving what worked — a **skill library**
(procedural memory): on success, store the solution; on a new task, retrieve and
**reuse** a matching skill instead of re-deriving it. No weight updates — it learns
by growing its memory (Voyager's key idea; builds on Units 2–3).

**From scratch:** a skill store that caches solutions and reuses them on repeat
tasks.

> 📚 **Resources:** [Voyager — Wang et al., 2023](https://arxiv.org/abs/2305.16291) ·
> [Generative Agents — Park et al., 2023](https://arxiv.org/abs/2304.03442)

In [ ]:
# From scratch — a skill library the agent reuses and grows.
skills = {}                                    # task-signature -> learned solution

def signature(task): return frozenset(re.findall(r"[a-z]+", task.lower())) - {"the","a","to","of"}

def solve(task, expensive_solver):
    sig = signature(task)
    for known_sig, sol in skills.items():
        if known_sig <= sig or sig <= known_sig:          # close enough -> reuse
            print(f"  ♻️  reused skill for '{task}'")
            return sol
    sol = expensive_solver(task)                           # first time: do the work
    skills[sig] = sol
    print(f"  🧠 learned new skill for '{task}'")
    return sol

solver = lambda t: f"<solution to: {t}>"
solve("sort a list in python", solver)
solve("reverse a string", solver)
solve("sort a list in python please", solver)             # retrieved, not re-solved
print("skills known:", len(skills))

In [ ]:
# SOTA — distil a reusable lesson with the model, store it, inject on similar tasks.
LESSONS = []
def run_with_lessons(task):
    hint = "\n".join(f"- {l}" for l in LESSONS) or "(none yet)"
    answer = chat([{"role": "system", "content": f"Relevant past lessons:\n{hint}"},
                   {"role": "user", "content": task}]).content
    # after a run, distil a one-line reusable lesson (procedural memory grows).
    lesson = chat([{"role": "system", "content": "In one short imperative sentence, state a reusable "
                    "lesson from solving this task."},
                   {"role": "user", "content": f"Task: {task}\nAnswer: {answer}"}]).content
    LESSONS.append(lesson.strip())
    return answer, lesson.strip()

ans, lesson = run_with_lessons("Write a robust retry wrapper.")
print("lesson stored:", lesson)

## Recap

Two axes of getting better — **weights** (slow, permanent) vs **context** (fast,
per-run):

| component | mechanism (from scratch) | SOTA packaging | axis |
|---|---|---|---|
| fine-tuning | gradient descent on a tiny classifier | hosted fine-tuning on JSONL | weights |
| RLHF / preferences | reward from winner>loser pairs (Bradley-Terry) | DPO / TRL / preference FT | weights |
| self-correction | draft → check → revise loop | model self-refine until judge approves | context |
| skill acquisition | reuse-or-learn skill library | distil & inject lessons (Voyager) | context |

**Course complete.** Across nine units, every agentic-AI component in `index.md`
(Sections 1, 3–10) reduced to the same pattern: **a few lines of plain Python that
reveal the mechanism, then the SOTA feature/library that packages it.** Tools and
MCP (Section 2) were Phase 1. You can now whiteboard any piece from first
principles *and* name the production tool for it — exactly the interview goal.

See `RESOURCES.md` for the consolidated reading list across all units.